# Open Notebook & Additional Resources

<a target="_blank" href="https://colab.research.google.com/github/Nicolepcx/ORM_AI_Agents_Bootcamp/blob/main/hands_on/DAY_2_HANDS_ON_Session_2_langgraph_episodic_procedural_tools.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>
<a target="_blank" href="https://learning.oreilly.com/library/view/ai-agents-the/0642572247775/">
  <img src="https://img.shields.io/badge/AI%20Agents%20Book-Read%20on%20O'Reilly-d40101?style=flat" alt="AI Agents Book – Read on O'Reilly"/>
</a>





#<font color="red" size="10">
<b>HANDS-ON TIME: 15 mins</b>
</font>

# About the Notebook

Hands-on: Episodic + procedural memory with tools

**Goal:** Extend a LangGraph agent so that:

1. **Episodic memory** — after each tool runs, persist the **tool name + arguments + result** (trimmed) into the LangGraph **store** so later turns can retrieve similar past tool use.
2. **Procedural memory** — expose a **`record_routine`** tool that saves a **named, reusable checklist** into the store (how you want the agent to behave for a recurring task).

You will use:

- `MessagesState` + `add_messages` + **checkpointer** (short-term / thread memory)
- **`InMemoryStore`** + `Runtime` + **`context_schema`** (long-term, cross-thread per `user_id`)
- A **ReAct-style loop**: `agent` → `tools` → **`persist_memories`** → back to `agent`

See also: `demos/DEMO_langgraph_memory_types.ipynb` for unified memory patterns.


In [1]:
%pip install -q "langgraph>=0.2" "langchain>=0.3" "langchain-openai>=0.3" "langchain-core>=0.3" "python-dotenv>=1.0"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 12.4 MB/s eta 0:00:00


In [3]:
from __future__ import annotations

import json
import os
import uuid
from dataclasses import dataclass
from datetime import datetime, timezone

from dotenv import load_dotenv
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.runnables import RunnableConfig
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
from langgraph.runtime import Runtime
from langgraph.store.memory import InMemoryStore

#load_dotenv()

NEBIUS_BASE_URL = "https://api.tokenfactory.nebius.com/v1/"
NEBIUS_API_KEY = os.getenv("NEBIUS_API_KEY", "")
if not NEBIUS_API_KEY:
    print("⚠️ NEBIUS_API_KEY not found.")
    NEBIUS_API_KEY = input("Enter your NEBIUS_API_KEY: ").strip()
NEBIUS_API_KEY = NEBIUS_API_KEY.strip().strip('"').strip("'")

NEBIUS_CHAT_MODEL = os.getenv("NEBIUS_CHAT_MODEL", "openai/gpt-oss-120b")
NEBIUS_EMBED_MODEL = os.getenv("NEBIUS_EMBED_MODEL", "Qwen/Qwen3-Embedding-8B")

llm: ChatOpenAI | None
if NEBIUS_API_KEY:
    llm = ChatOpenAI(
        model=NEBIUS_CHAT_MODEL,
        api_key=NEBIUS_API_KEY,
        base_url=NEBIUS_BASE_URL,
        temperature=0,
    )
else:
    llm = None


def _stub_embed(texts: list[str]) -> list[list[float]]:
    out: list[list[float]] = []
    for t in texts:
        vec = [float((ord(c) % 17) / 17.0) for c in t[:24]]
        while len(vec) < 8:
            vec.append(0.0)
        out.append(vec[:8])
    return out


def build_store() -> InMemoryStore:
    if NEBIUS_API_KEY:
        embedder = OpenAIEmbeddings(
            model=NEBIUS_EMBED_MODEL,
            api_key=NEBIUS_API_KEY,
            base_url=NEBIUS_BASE_URL,
            tiktoken_enabled=False,
        )
        dims = len(embedder.embed_query("dimension probe"))
        return InMemoryStore(
            index={
                "embed": embedder,
                "dims": dims,
                "fields": ["tool_name", "summary", "text", "routine_name", "steps", "$"],
            }
        )
    return InMemoryStore(index={"embed": _stub_embed, "dims": 8})


@dataclass(frozen=True)
class AppContext:
    user_id: str


def ns_episodic(ctx: AppContext) -> tuple[str, str]:
    return (ctx.user_id, "tool_episodes")


def ns_procedural(ctx: AppContext) -> tuple[str, str]:
    return (ctx.user_id, "routines")


def _iso() -> str:
    return datetime.now(timezone.utc).replace(microsecond=0).isoformat()


@tool
def multiply_ints(a: int, b: int) -> int:
    """Multiply two integers. Use for quick arithmetic."""
    return a * b


@tool
def to_snake_case(text: str) -> str:
    """Convert a short phrase to snake_case (letters and spaces only)."""
    cleaned = "".join(c.lower() if c.isalnum() or c.isspace() else " " for c in text)
    return "_".join(cleaned.split())


@tool
def record_routine(name: str, steps: str) -> str:
    """Save a named reusable routine (numbered steps) for this user."""
    return json.dumps({"ok": True, "routine": name, "saved_steps_preview": steps[:120]})


TOOLS = [multiply_ints, to_snake_case, record_routine]
tool_node = ToolNode(TOOLS)


def _last_ai_with_tool_calls(msgs: list[BaseMessage]) -> AIMessage | None:
    for m in reversed(msgs):
        if isinstance(m, AIMessage) and (m.tool_calls or []):
            return m
    return None


def _tool_results_after_ai(msgs: list[BaseMessage], ai: AIMessage) -> list[ToolMessage]:
    idx = msgs.index(ai)
    out: list[ToolMessage] = []
    for j in range(idx + 1, len(msgs)):
        if isinstance(msgs[j], ToolMessage):
            out.append(msgs[j])
        else:
            break
    return out


def persist_memories(state: MessagesState, runtime: Runtime[AppContext]) -> dict:
    """Episodic: each ordinary tool call + result. Procedural: record_routine → store."""
    msgs = state["messages"]
    ai = _last_ai_with_tool_calls(msgs)
    if ai is None:
        return {}
    ctx = runtime.context
    tool_results = _tool_results_after_ai(msgs, ai)
    if not tool_results:
        return {}

    calls = list(ai.tool_calls or [])
    by_id = {tm.tool_call_id: tm for tm in tool_results}

    for call in calls:
        tid = call.get("id") or call.get("tool_call_id")
        name = call.get("name")
        args = call.get("args") or {}
        tm = by_id.get(tid)
        if tm is None:
            continue

        if name == "record_routine":
            routine_name = str(args.get("name", "unnamed"))
            steps = str(args.get("steps", ""))
            runtime.store.put(
                ns_procedural(ctx),
                f"routine_{routine_name}",
                {
                    "routine_name": routine_name,
                    "steps": steps,
                    "updated_at": _iso(),
                },
            )
            continue

        summary = f"{name}({args}) -> {str(tm.content)[:500]}"
        runtime.store.put(
            ns_episodic(ctx),
            str(uuid.uuid4()),
            {
                "tool_name": name,
                "args": args,
                "result_preview": str(tm.content)[:1500],
                "summary": summary,
                "created_at": _iso(),
            },
        )

    return {}


def build_system_prompt(runtime: Runtime[AppContext], user_query: str) -> str:
    ctx = runtime.context
    epi = list(
        runtime.store.search(
            ns_episodic(ctx),
            query=user_query or "recent tool usage",
            limit=4,
        )
    )
    proc_items = list(runtime.store.search(ns_procedural(ctx), limit=20))

    epi_lines = [f"- {it.value.get('summary', it.value)}" for it in epi]
    episodic_block = "\n".join(epi_lines) if epi_lines else "(no tool episodes yet)"

    proc_lines = []
    for it in proc_items:
        v = it.value
        proc_lines.append(f"**{v.get('routine_name', it.key)}**:\n{v.get('steps', v)}")
    proc_block = "\n\n".join(proc_lines) if proc_lines else "(no saved routines yet)"

    return (
        "You are a helpful assistant with tools. Use tools when they help. "
        "When the user wants a repeatable workflow remembered, call record_routine.\n\n"
        f"Relevant past tool episodes:\n{episodic_block}\n\n"
        f"Saved procedural routines:\n{proc_block}"
    )


def agent_node(state: MessagesState, runtime: Runtime[AppContext]) -> dict:
    if llm is None:
        return {
            "messages": [
                AIMessage(
                    content="Set NEBIUS_API_KEY to run the tool-calling loop with this notebook."
                )
            ]
        }

    h = next((m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), None)
    uq = (h.content if h and isinstance(h.content, str) else "") or ""

    sys = SystemMessage(content=build_system_prompt(runtime, uq))
    bound = llm.bind_tools(TOOLS)
    reply = bound.invoke([sys, *state["messages"]])
    return {"messages": [reply]}


def route_after_agent(state: MessagesState) -> str:
    last = state["messages"][-1]
    if isinstance(last, AIMessage) and last.tool_calls:
        return "tools"
    return END


workflow = StateGraph(MessagesState, context_schema=AppContext)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", tool_node)
workflow.add_node("persist_memories", persist_memories)

workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", route_after_agent, {"tools": "tools", END: END})
workflow.add_edge("tools", "persist_memories")
workflow.add_edge("persist_memories", "agent")

CHECKPOINTER = InMemorySaver()
STORE = build_store()
app = workflow.compile(checkpointer=CHECKPOINTER, store=STORE)
print("Compiled: agent → tools → persist_memories → agent")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Compiled: agent → tools → persist_memories → agent


## Try it

Use the same `user_id` across turns so episodic and procedural rows accumulate in the store.


In [4]:
def run_turn(thread_id: str, user_id: str, text: str) -> dict:
    cfg: RunnableConfig = {"configurable": {"thread_id": thread_id}}
    return app.invoke(
        {"messages": [HumanMessage(content=text, id=str(uuid.uuid4()))]},
        cfg,
        context=AppContext(user_id=user_id),
    )


USER = "ai-agent-bootcamp-42"
THREAD = "session-tools-1"

out1 = run_turn(THREAD, USER, "What is 21 times 7? Use the multiply_ints tool.")
print(out1["messages"][-1].content)

out2 = run_turn(THREAD, USER, "Convert 'Hello World' to snake_case using the tool.")
print(out2["messages"][-1].content)

run_turn(
    THREAD,
    USER,
    'Call record_routine with name="release_checklist" and steps="1) Run tests 2) Bump version 3) Tag git 4) Publish notes"',
)
out3 = run_turn(THREAD, USER, "Summarize my release_checklist routine.")
print(out3["messages"][-1].content)


The product of 21 and 7 is **147**.
The snake_case version of “Hello World” is:

**hello_world**
**Release Checklist Summary**

Your **release_checklist** routine outlines the essential steps to perform before publishing a new software version:

1. **Run tests** – Verify that the codebase passes all automated (and any required manual) tests.  
2. **Bump version** – Increment the project’s version number according to your versioning scheme (e.g., semantic versioning).  
3. **Tag git** – Create a Git tag that marks the new version in the repository, making it easy to reference the exact commit.  
4. **Publish notes** – Draft and release the release notes (changelog, documentation updates, etc.) so users know what’s changed.  

Following these four steps ensures a smooth, reliable release process.


## Inspect the store

Episodic tool rows: namespace `(user_id, "tool_episodes")`. Routines: `(user_id, "routines")`.


In [5]:
ctx = AppContext(user_id=USER)
print("Episodic keys:", [i.key for i in STORE.search(ns_episodic(ctx), limit=5)])
print("Routine keys:", [i.key for i in STORE.search(ns_procedural(ctx), limit=5)])
for it in STORE.search(ns_procedural(ctx), limit=3):
    print(it.key, "=>", str(it.value.get("steps", it.value))[:220])


Episodic keys: ['fafb5dc0-0259-4229-bb34-2dc8ece97c9d', 'a78c7a9b-afc8-43fa-a0f1-5d087756f3bd']
Routine keys: ['routine_release_checklist']
routine_release_checklist => 1) Run tests 2) Bump version 3) Tag git 4) Publish notes
